<a href="https://colab.research.google.com/github/naamasarshalom-art/segmentation_cellpose/blob/main/1_segmentation_cellpose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nucleus Segmentation with Cellpose

This notebook performs nucleus segmentation on fluorescence microscopy images using the Cellpose model.

**Input:** A folder containing `.tif` or `.nd2` images with 2 fluorescence channels:
- Channel 0 — Blue (DAPI)
- Channel 1 — Red (LAMINB1)

**Output per image:**
- Individual cropped `.jpg` files for each detected nucleus
- A full overview `.jpg` with bounding boxes drawn around all detected nuclei

**Pipeline steps:**
1. Mount Google Drive and install dependencies
2. Load the Cellpose model (GPU required)
3. For each image: read → build weighted input → segment → crop → save

## 1. Setup — Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install git+https://www.github.com/mouseland/cellpose.git
!pip install nd2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Cloning https://www.github.com/mouseland/cellpose.git to /tmp/pip-req-build-f8hw8g4o
  Running command git clone --filter=blob:none --quiet https://www.github.com/mouseland/cellpose.git /tmp/pip-req-build-f8hw8g4o
  Resolved https://www.github.com/mouseland/cellpose.git to commit bb881c5e27881155c8fb3626f198e97101c47514
  Preparing metadata (setup.py) ... done


## 2. Imports

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
import imageio.v2 as imageio
import nd2

from pathlib import Path
from natsort import natsorted

from scipy import ndimage
from cellpose import models, core, io

## 3. Configuration

Set all paths and parameters here — nothing else in this notebook needs to be changed between runs.

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_DIR   = Path("/content/drive/MyDrive/model_nuc/batch 1 - 20.10.25/123-test")
OUTPUT_DIR  = Path("/content/drive/MyDrive/model_nuc/test")

# ── Supported formats ──────────────────────────────────────────────────────
SUPPORTED_EXTENSIONS = [".tif", ".nd2"]

# ── Weighted input for segmentation: weighted = alpha*ch0 + beta*ch1 ───────
# Channel 0 (blue) and Channel 1 (red) are blended before
# being passed to Cellpose. Adjust these weights based on your staining.
ALPHA = 0.3   # weight for channel 0 (blue)
BETA  = 0.7   # weight for channel 1 (red)

# ── Crop padding: pixels added around each nucleus bounding box ─────────────
PADDING = 20

# ── Cellpose model parameters ──────────────────────────────────────────────
FLOW_THRESHOLD    = 0.2
CELLPROB_THRESHOLD = 0.0
BATCH_SIZE        = 32

## 4. Load Cellpose Model

Requires GPU. If this cell raises an error, go to Runtime → Change runtime type → GPU.

In [4]:
io.logger_setup()

if not core.use_gpu():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU.")

model = models.CellposeModel(gpu=True)
print("Cellpose model loaded successfully on GPU.")

creating new log file
[GUI INFO] : WRITING LOG OUTPUT TO /root/.cellpose/run.log

cellpose version: 	4.1.2.dev50+gbb881c5e2 
platform:       	linux 
python version: 	3.12.13 
torch version:  	2.10.0+cu128
2026-05-14 07:10:52,784 [io INFO] WRITING LOG OUTPUT TO /root/.cellpose/run.log
2026-05-14 07:10:52,784 [io INFO] 
cellpose version: 	4.1.2.dev50+gbb881c5e2 
platform:       	linux 
python version: 	3.12.13 
torch version:  	2.10.0+cu128
2026-05-14 07:10:53,190 [core INFO] ** TORCH CUDA version installed and working. **
2026-05-14 07:10:53,193 [core INFO] ** TORCH CUDA version installed and working. **
2026-05-14 07:10:53,194 [core INFO] >>>> using GPU (CUDA)
2026-05-14 07:10:55,674 [models INFO] Downloading: "https://huggingface.co/mouseland/cellpose-sam/resolve/main/cpsam" to /root/.cellpose/models/cpsam



100%|██████████| 1.15G/1.15G [00:07<00:00, 170MB/s]


Cellpose model loaded successfully on GPU.


## 5. Helper Functions

In [5]:
def normalize_to_uint8(x):
    """
    Normalize a 2D array to the range [0, 255] and cast to uint8.
    A small epsilon (1e-8) prevents division by zero on uniform images.
    """
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)
    return (x * 255).astype(np.uint8)


def read_image(file_path):
    """
    Read a microscopy image from a .tif/.tiff or .nd2 file.

    Returns the image as a numpy array in (H, W, C) layout.
    Input files are assumed to be in (C, H, W) layout (standard for
    multi-channel fluorescence images), so the channel axis is moved last.
    """
    ext = file_path.suffix.lower()

    if ext in [".tif", ".tiff"]:
        with tiff.TiffFile(file_path) as f:
            img = f.asarray()          # shape: (C, H, W)
    elif ext == ".nd2":
        with nd2.ND2File(file_path) as f:
            img = f.asarray()          # shape: (C, H, W)
    else:
        raise ValueError(f"Unsupported file format: {ext}")

    # Convert (C, H, W) → (H, W, C)
    img_hwc = np.moveaxis(img, 0, -1)
    return img_hwc


def build_weighted_input(img_hwc, alpha=ALPHA, beta=BETA):
    """
    Build the single-channel input for Cellpose by blending two fluorescence
    channels: weighted = alpha * ch0 + beta * ch1.

    Cellpose expects input shape (H, W, 1) when given a single channel.
    """
    weighted = alpha * img_hwc[:, :, 0] + beta * img_hwc[:, :, 1]
    return weighted[..., np.newaxis]   # shape: (H, W, 1)


def build_rgb_display(img_hwc):
    """
    Build a uint8 RGB image for display and saving:
    - Red channel  ← fluorescence channel 1
    - Blue channel ← fluorescence channel 0
    - Green channel = 0 (unused)
    """
    H, W = img_hwc.shape[:2]
    rgb = np.zeros((H, W, 3), dtype=np.uint8)
    rgb[:, :, 0] = normalize_to_uint8(img_hwc[:, :, 1])   # red
    rgb[:, :, 2] = normalize_to_uint8(img_hwc[:, :, 0])   # blue
    return rgb

## 6. Core Processing Function

In [6]:
def process_image(file_path, output_dir, alpha=ALPHA, beta=BETA, padding=PADDING):
    """
    Run the full segmentation pipeline on a single image file.

    Steps:
      1. Read image (.tif or .nd2) → (H, W, C) array
      2. Build weighted single-channel input for Cellpose
      3. Run Cellpose segmentation to get nucleus masks
      4. Find connected components (one per nucleus)
      5. For each nucleus: crop, build RGB, save as JPG
      6. Save full image with all bounding boxes as JPG

    Args:
        file_path  (Path): Path to the input image file.
        output_dir (Path): Directory where outputs will be saved.
        alpha      (float): Weight for channel 0 in the blended input.
        beta       (float): Weight for channel 1 in the blended input.
        padding    (int): Pixels to add around each nucleus bounding box.

    Returns:
        int: Number of nuclei detected and saved.
    """
    print(f"\nProcessing: {file_path.name}")

    # ── Step 1: Read image ────────────────────────────────────────────────
    img_hwc = read_image(file_path)
    H, W = img_hwc.shape[:2]
    print(f"  Image size: {H}x{W}, channels: {img_hwc.shape[2]}")

    # ── Step 2: Build weighted input ──────────────────────────────────────
    img_weighted = build_weighted_input(img_hwc, alpha, beta)

    # ── Step 3: Run Cellpose segmentation ─────────────────────────────────
    print("  Running Cellpose...")
    masks, flows, styles = model.eval(
        img_weighted,
        batch_size=BATCH_SIZE,
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
        normalize={"tile_norm_blocksize": 0}
    )

    # ── Step 4: Find connected components (one per nucleus) ───────────────
    binary_mask = masks > 0
    labeled_cc, num_cc = ndimage.label(binary_mask)
    print(f"  Detected {num_cc} nuclei")

    # ── Step 5: Crop and save each nucleus ────────────────────────────────
    bbox_list = []

    for cc_label in range(1, num_cc + 1):
        component_mask = labeled_cc == cc_label
        ys, xs = np.where(component_mask)
        if len(xs) == 0:
            continue

        # Compute bounding box with padding, clamped to image boundaries
        x1 = max(0, xs.min() - padding)
        y1 = max(0, ys.min() - padding)
        x2 = min(W - 1, xs.max() + padding)
        y2 = min(H - 1, ys.max() + padding)

        # Crop each channel and build RGB
        crop_hwc = img_hwc[y1:y2+1, x1:x2+1, :]
        crop_rgb = build_rgb_display(crop_hwc)

        # Save crop as JPG; filename encodes source image and bounding box
        filename = f"{file_path.stem}.{x1}_{y1}_{x2}_{y2}.jpg"
        save_path = output_dir / filename
        imageio.imwrite(str(save_path), crop_rgb)

        bbox_list.append((x1, y1, x2, y2, filename))

    print(f"  Saved {len(bbox_list)} nucleus crops")

    # ── Step 6: Save full overview image with bounding boxes ──────────────
    full_rgb = build_rgb_display(img_hwc)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(full_rgb)
    ax.set_title(f"{file_path.name} — {num_cc} nuclei detected")

    for x1, y1, x2, y2, fname in bbox_list:
        ax.add_patch(plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            edgecolor='yellow', facecolor='none', linewidth=1.5
        ))
        ax.text(x1, y1 - 5, fname.split(".")[-2],
                color='yellow', fontsize=6)

    ax.axis('off')
    overview_path = output_dir / f"{file_path.stem}_overview.jpg"
    fig.savefig(str(overview_path), bbox_inches='tight', pad_inches=0, dpi=300)
    plt.close(fig)
    print(f"  Overview saved: {overview_path.name}")

    return len(bbox_list)

## 7. Run — Process All Images in the Input Folder

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Collect all supported files, sorted naturally (pic1, pic2, ..., pic10)
files = natsorted([
    f for f in INPUT_DIR.iterdir()
    if f.suffix.lower() in SUPPORTED_EXTENSIONS
])

if not files:
    raise FileNotFoundError(
        f"No supported images found in {INPUT_DIR}\n"
        f"Supported formats: {SUPPORTED_EXTENSIONS}"
    )

print(f"Found {len(files)} image(s) to process")
print(f"Output directory: {OUTPUT_DIR}\n")

# ── Main loop ─────────────────────────────────────────────────────────────
errors = []
total_nuclei = 0

for file_path in files:
    # Skip if overview already exists (allows resuming interrupted runs)
    expected_overview = OUTPUT_DIR / f"{file_path.stem}_overview.jpg"
    if expected_overview.exists():
        print(f"Skipping {file_path.name} (already processed)")
        continue

    try:
        n = process_image(file_path, OUTPUT_DIR)
        total_nuclei += n
    except Exception as e:
        print(f"ERROR processing {file_path.name}: {e}")
        errors.append(file_path.name)

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "="*50)
print(f"Done. Processed {len(files) - len(errors)} / {len(files)} images.")
print(f"Total nuclei saved: {total_nuclei}")

if errors:
    print(f"\nFailed files ({len(errors)}):")
    for e in errors:
        print(f"  - {e}")

Found 10 image(s) to process
Output directory: /content/drive/MyDrive/model_nuc/test


Processing: עותק של pic 1 - 100X.nd2
  Image size: 1608x1608, channels: 2
  Running Cellpose...
  Detected 5 nuclei
  Saved 5 nucleus crops
  Overview saved: עותק של pic 1 - 100X_overview.jpg

Processing: עותק של pic 1.tif
  Image size: 1608x1608, channels: 2
  Running Cellpose...
  Detected 2 nuclei
  Saved 2 nucleus crops
  Overview saved: עותק של pic 1_overview.jpg

Processing: עותק של pic 10- 100X.nd2
  Image size: 1608x1608, channels: 2
  Running Cellpose...
  Detected 4 nuclei
  Saved 4 nucleus crops
  Overview saved: עותק של pic 10- 100X_overview.jpg

Processing: עותק של pic 11.tif
  Image size: 1608x1608, channels: 2
  Running Cellpose...
  Detected 3 nuclei
  Saved 3 nucleus crops
  Overview saved: עותק של pic 11_overview.jpg

Processing: עותק של pic 100-100x.nd2
  Image size: 1608x1608, channels: 2
  Running Cellpose...
  Detected 3 nuclei
  Saved 3 nucleus crops
  Overview saved: עותק של p